# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

This notebook implements the Retrieval-Augmented Generation pipeline,
including hybrid retrieval and answer generation.

In [1]:
# Install dependencies

!pip install sentence-transformers
!pip install transformers
!pip install rank-bm25
!pip install accelerate

In [2]:
# Imports

import json
import math
from pathlib import Path
from dataclasses import dataclass
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

C:\Users\gh300\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Paths and data loading

DATA_DIR = Path(".")

corpus_path = DATA_DIR / "background_corpus.json"
query_path = DATA_DIR / "benchmark_input_only.json"
output_path = DATA_DIR / "test_outputs.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(query_path, "r", encoding="utf-8") as f:
    query_payload = json.load(f)

queries = query_payload["queries"]

print(f"Loaded {len(corpus)} documents and {len(queries)} queries")

Loaded 17 documents and 13 queries


In [4]:
# Dataclasses

@dataclass
class Chunk:
    doc_id: str
    title: str
    section: str
    text: str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

In [5]:
# Chunking function

def chunk_documents(corpus_docs):
    chunks = []

    for doc in corpus_docs:
        doc_id = doc.get("doc_id", "")
        title = doc.get("title", "")
        sections = doc.get("sections", [])

        if sections:
            for sec in sections:
                heading = sec.get("heading", "Section")
                content_list = sec.get("content", [])
                text = " ".join(content_list).strip()
                if text:
                    chunks.append(Chunk(
                        doc_id=doc_id,
                        title=title,
                        section=heading,
                        text=text
                    ))
        else:
            full_text = doc.get("full_text", "").strip()
            if full_text:
                chunks.append(Chunk(
                    doc_id=doc_id,
                    title=title,
                    section="Full text",
                    text=full_text
                ))

    return chunks

chunks = chunk_documents(corpus)
print(f"Produced {len(chunks)} chunks")

Produced 106 chunks


In [6]:
# Load embedding model

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1557.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [7]:
# Embed chunks

chunk_texts = [f"[{c.section}] {c.text}" for c in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Chunk embeddings shape:", chunk_embeddings.shape)

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.02it/s]

Chunk embeddings shape: (106, 384)


In [8]:
# Build BM25 retriever

tokenized_chunks = [text.lower().split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 retriever ready")

BM25 retriever ready


In [9]:
# Dense retrieval function

def dense_retrieve(query, top_k=5):
    query_emb = embedding_model.encode([query], convert_to_numpy=True)[0]

    # cosine similarity
    chunk_norms = np.linalg.norm(chunk_embeddings, axis=1)
    query_norm = np.linalg.norm(query_emb)
    sims = (chunk_embeddings @ query_emb) / (chunk_norms * query_norm + 1e-12)

    top_indices = np.argsort(sims)[::-1][:top_k]

    return [
        RetrievalResult(chunk=chunks[i], score=float(sims[i]))
        for i in top_indices
    ]

In [10]:
# BM25 retrieval function

def bm25_retrieve(query, top_k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        RetrievalResult(chunk=chunks[i], score=float(scores[i]))
        for i in top_indices
    ]

In [11]:
# Hybrid retrieval function

def hybrid_retrieve(query, top_k=5, rrf_k=60):
    dense_results = dense_retrieve(query, top_k=10)
    bm25_results = bm25_retrieve(query, top_k=10)

    score_map = {}

    for rank, res in enumerate(dense_results, start=1):
        key = (res.chunk.doc_id, res.chunk.section, res.chunk.text)
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (rrf_k + rank)

    for rank, res in enumerate(bm25_results, start=1):
        key = (res.chunk.doc_id, res.chunk.section, res.chunk.text)
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (rrf_k + rank)

    merged = [
        RetrievalResult(chunk=v["chunk"], score=v["score"])
        for v in score_map.values()
    ]

    merged.sort(key=lambda x: x.score, reverse=True)
    return merged[:top_k]

In [12]:
# Load Qwen model

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

print("Qwen loaded")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 290/290 [00:00<00:00, 1117.22it/s]


Qwen loaded


In [13]:
# Prompt builder

def build_messages(query, retrieved_results):
    context = "\n\n".join(
        [f"[{r.chunk.section}] {r.chunk.text}" for r in retrieved_results]
    )

    return [
        {
            "role": "system",
            "content": (
                "You are a South Asian cuisine assistant. "
                "Answer only using the retrieved context. "
                "Do not guess or add outside knowledge. "
                "If the context is insufficient, say that the context does not clearly say."
            )
        },
        {
            "role": "user",
            "content": (
                f"Retrieved context:\n{context}\n\n"
                f"Question: {query}\n\n"
                "Write a short answer in 1 to 3 sentences. "
                "Do not repeat the instructions. "
                "Do not mention 'context' or 'documents' in your answer."
            )
        }
    ]

In [14]:
# Generation function

def generate_answer(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response.strip()

In [15]:
# Test on one query first

sample_query = queries[0]["query"]
sample_results = hybrid_retrieve(sample_query, top_k=5)
sample_messages = build_messages(sample_query, sample_results)
sample_answer = generate_answer(sample_messages)

print("QUERY:", sample_query)
print()
print("ANSWER:", sample_answer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUERY: What is biryani and which region is it originally associated with?

ANSWER: Biryani is a rice dish made with rice, spices, and usually meat, originating from South Asia.


In [16]:
# Full inference loop

results = []

for item in queries:
    query_id = item["query_id"]
    query = item["query"]

    retrieved = hybrid_retrieve(query, top_k=5)
    messages = build_messages(query, retrieved)
    response = generate_answer(messages)

    results.append({
        "query_id": query_id,
        "query": query,
        "response": response,
        "retrieved_context": [
            {
                "doc_id": r.chunk.doc_id,
                "text": f"[{r.chunk.section}] {r.chunk.text[:500]}"
            }
            for r in retrieved
        ]
    })

print(f"Generated answers for {len(results)} queries")

Generated answers for 13 queries


In [17]:
# Save outputs

with open(output_path, "w", encoding="utf-8") as f:
    json.dump({"results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved outputs to {output_path}")

Saved outputs to test_outputs.json


In [21]:
# Demo: test custom query and inspect retrieved context

custom_query = input("Enter your question: ")

retrieved = hybrid_retrieve(custom_query, top_k=5)
messages = build_messages(custom_query, retrieved)
response = generate_answer(messages)

print("\nANSWER:")
print(response)

print("\nRETRIEVED CONTEXT:")
for i, r in enumerate(retrieved, 1):
    print(f"\n{i}. [{r.chunk.doc_id} | {r.chunk.section}]")
    print(r.chunk.text[:300])

Enter your question:  what are momos



ANSWER:
Momsos are savory pastries typically round shaped and filled with spices. They originated in South Asian cuisine and are popular in many countries including India, Nepal, and Sri Lanka.

RETRIEVED CONTEXT:

1. [wiki_cardamom | Introduction]
Cardamom ( / ˈ k ɑːr d ə m ə m / ), sometimes cardamon or cardamum , is a spice made from the seeds of several plants in the genera Elettaria and Amomum in the family Zingiberaceae . Both genera are native to the Indian subcontinent and Indonesia . They are recognized by their small seed pods: trian

2. [wiki_samosa | Introduction]
Samosa is a savory pastry, usually triangular, that is fried or baked and filled with spiced ingredients. It is one of the best-known snacks in South Asian cuisine.

3. [wiki_nepali_cuisine | Regional diversity]
Food differs between the plains, hills, and Himalayan regions because of climate, agriculture, and local traditions. Ingredients and preparation methods change according to what is available locally.

4. 

In [19]:
# Preview outputs

results[:2]

[{'query_id': 'sa_000',
  'query': 'What is biryani and which region is it originally associated with?',
  'response': 'Biryani is a rice dish made with rice, spices, and usually meat, originating from South Asia.',
  'retrieved_context': [{'doc_id': 'wiki_biryani',
    'text': '[Introduction] Biryani is a rice dish made with rice, spices, and usually meat, although vegetarian versions also exist. It is one of the best-known dishes associated with South Asian cuisine.'},
   {'doc_id': 'wiki_biryani',
    'text': '[Regional forms] There are many regional varieties of biryani across the Indian subcontinent. The dish is popular in everyday dining as well as festivals and celebrations.'},
   {'doc_id': 'wiki_biryani',
    'text': '[Preparation] Biryani is often prepared by layering rice with spiced ingredients and finishing the cooking together. Different styles vary in seasoning, assembly, and cooking technique.'},
   {'doc_id': 'wiki_nepali_cuisine',
    'text': '[Regional diversity] Foo